In [1]:
import pandas as pd
import sqlite3

inst = pd.read_csv("osap_rap_2020_institutions_clean.csv")
prog = pd.read_csv("osap_rap_2020_programs_clean.csv")

conn = sqlite3.connect("osap.db")
inst.to_sql("institutions", conn, if_exists="replace", index=False)
prog.to_sql("programs", conn, if_exists="replace", index=False)

def sql(query):
    return pd.read_sql(query, conn)

In [2]:
sql("SELECT COUNT(*) AS schools FROM institutions")

,schools
0,235


In [3]:
sql("""
SELECT sector, COUNT(*) AS schools
FROM institutions
GROUP BY sector
""")


,sector,schools
0,College,24
1,Other,23
2,Private Career College,164
3,University,24


In [4]:
sql("""
SELECT sector,
       SUM(loan_recipients) AS recipients,
       SUM(rap_users) AS rap_users,
       ROUND(100.0 * SUM(rap_users) / SUM(loan_recipients), 1) AS rap_rate_pct
FROM institutions
GROUP BY sector
ORDER BY rap_rate_pct DESC
""")


,sector,recipients,rap_users,rap_rate_pct
0,Other,1018.0,330.0,32.4
1,Private Career College,14377.0,4073.0,28.3
2,University,60228.0,16495.0,27.4
3,College,61093.0,16351.0,26.8


In [5]:
sql("""
SELECT institution, sector, loan_recipients,
       ROUND(100 * rap_rate, 1) AS rap_rate_pct
FROM institutions
WHERE loan_recipients >= 100
ORDER BY rap_rate DESC
LIMIT 10
""")

,institution,sector,loan_recipients,rap_rate_pct
0,CANADIAN COLLEGE OF NATUROPATHIC MED,Other,102.0,60.8
1,CANADIAN MEMORIAL CHIROPRACTIC COLLEGE,Other,129.0,43.4
2,TRIOS COLLEGE - SCARBOROUGH,Private Career College,144.0,41.0
3,ONTARIO COLLEGE OF ART AND DESIGN,University,546.0,40.8
4,"ANDERSON COLLEGE OF HEALTH, BUSINESS & TECHNOL...",Private Career College,218.0,40.4
5,ACADEMY OF LEARNING - TORONTO/ALBION,Private Career College,100.0,40.0
6,TYNDALE UNIVERSITY,Other,178.0,39.9
7,TRIOS COLLEGE - TORONTO,Private Career College,396.0,37.1
8,CANADIAN BUSINESS COLLEGE - MISSISSAUGA,Private Career College,127.0,37.0
9,UNIVERSITY OF ONTARIO INST OF TECHNOLOGY,University,1666.0,36.9


In [6]:
sql("""
SELECT institution, loan_recipients, rap_users,
       ROUND(100 * rap_rate, 1) AS rap_rate_pct
FROM institutions
WHERE institution IN ('LAKEHEAD UNIVERSITY', 'CONFEDERATION COLLEGE')

UNION ALL

SELECT 'ONTARIO AVERAGE', SUM(loan_recipients), SUM(rap_users),
       ROUND(100.0 * SUM(rap_users) / SUM(loan_recipients), 1)
FROM institutions
""")

,institution,loan_recipients,rap_users,rap_rate_pct
0,LAKEHEAD UNIVERSITY,1183.0,295.0,24.9
1,CONFEDERATION COLLEGE,691.0,131.0,19.0
2,ONTARIO AVERAGE,136716.0,37249.0,27.2


In [7]:
sql("""
SELECT program,
       SUM(loan_recipients) AS recipients,
       ROUND(100.0 * SUM(rap_users) / SUM(loan_recipients), 1) AS rap_rate_pct
FROM programs
WHERE sector = 'University'
GROUP BY program
HAVING SUM(loan_recipients) >= 500
ORDER BY rap_rate_pct DESC
""")

,program,recipients,rap_rate_pct
0,Fine & Applied Arts,1778.0,36.7
1,Food Science & Nutrition,638.0,33.9
2,Humanities,1338.0,32.9
3,Social Sciences,6128.0,32.5
4,Other Arts and Science,19653.0,32.5
5,Agricultural & Biological Sciences,1246.0,30.0
6,Other Health Professions,1098.0,28.3
7,Other Education,554.0,28.3
8,Nursing,2596.0,26.5
9,"Kinesiology, Recreation & Physical Education",1349.0,26.5


In [8]:
sql("""
SELECT RANK() OVER (ORDER BY rap_rate DESC) AS rank,
       institution, loan_recipients,
       ROUND(100 * rap_rate, 1) AS rap_rate_pct
FROM institutions
WHERE sector = 'College'
ORDER BY rank
""")

,rank,institution,loan_recipients,rap_rate_pct
0,1,CENTENNIAL COLLEGE,3235.0,35.5
1,2,SENECA COLLEGE,5254.0,30.3
2,3,GEORGE BROWN COLLEGE,5739.0,29.7
3,4,HUMBER COLLEGE,6304.0,29.2
4,5,SHERIDAN COLLEGE,3953.0,28.3
5,6,LOYALIST COLLEGE,935.0,27.4
6,7,DURHAM COLLEGE,2924.0,27.1
7,8,ST. CLAIR COLLEGE,2268.0,26.5
8,9,CANADORE COLLEGE,730.0,25.9
9,10,ST. LAWRENCE COLLEGE,1797.0,25.4
